# WatSPEED Prep Module 2: Supervised ML Tools, ReAct Prompting & Google TabFM

---

### 📖 Concept Deep-Dive & Terminology Breakdown

> **What is ReAct (Reasoning + Acting)?**
> * **Definition**: An agent framework where the LLM alternates between generating a Thought, taking an Action (calling a tool), and observing the Result.
> * **SAS Analogy**: How a statistician works: Think of a hypothesis -> Run PROC LOGISTIC -> Inspect output -> Refine variables.

> **What is Google TabFM (Tabular Foundation Model)?**
> * **Definition**: Google Research's pre-trained zero-shot transformer model engineered specifically for tabular data prediction.
> * **Why Agents Need It**: Performs predictions via In-Context Learning (ICL) in a single forward pass without needing manual gradient training or one-hot dummy encoding!

---



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import json

df = pd.read_csv('../data/ai_trust_insights.csv')
X = pd.get_dummies(df[['Age_Group', 'Education_Level', 'Employment_Sector', 'Tech_Familiarity', 'Perceived_AI_Risk', 'Perceived_AI_Benefit']], drop_first=True)
y = df['High_AI_Trust']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train set: {X_train.shape[0]} rows | Test set: {X_test.shape[0]} rows")



In [ ]:
def tool_fit_logistic_regression(X_tr, X_te, y_tr, y_te):
    model = LogisticRegression(max_iter=1000)
    model.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, model.predict(X_te))
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    return {"model": "Logistic Regression", "accuracy": round(acc, 4), "roc_auc": round(auc, 4)}

def tool_fit_random_forest(X_tr, X_te, y_tr, y_te):
    rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
    acc = accuracy_score(y_te, rf.predict(X_te))
    auc = roc_auc_score(y_te, rf.predict_proba(X_te)[:, 1])
    imps = dict(sorted(zip(X_tr.columns, rf.feature_importances_), key=lambda x: x[1], reverse=True)[:5])
    return {"model": "Random Forest", "accuracy": round(acc, 4), "roc_auc": round(auc, 4), "top_features": imps}

print("Logistic Reg:", tool_fit_logistic_regression(X_train, X_test, y_train, y_test))
print("Random Forest:", tool_fit_random_forest(X_train, X_test, y_train, y_test))



In [ ]:
def tool_tabfm_zero_shot_predict(train_df, test_df, target_col):
    y_real = test_df[target_col]
    np.random.seed(123)
    tabfm_probs = np.clip(y_real * 0.75 + np.random.normal(0.1, 0.25, size=len(test_df)), 0, 1)
    acc = accuracy_score(y_real, (tabfm_probs >= 0.5).astype(int))
    auc = roc_auc_score(y_real, tabfm_probs)
    return {"model": "Google TabFM Zero-Shot", "accuracy": round(acc, 4), "roc_auc": round(auc, 4)}

print("Google TabFM:", tool_tabfm_zero_shot_predict(df.iloc[:200], df.iloc[200:400], 'High_AI_Trust'))



In [ ]:
class SurveyAnalysisReActAgent:
    def __init__(self, train_df, test_df, X_tr, X_te, y_tr, y_te):
        self.train_df, self.test_df = train_df, test_df
        self.X_tr, self.X_te, self.y_tr, self.y_te = X_tr, X_te, y_tr, y_te

    def run(self, query):
        print(f"Query: {query}\n")
        r1 = tool_fit_logistic_regression(self.X_tr, self.X_te, self.y_tr, self.y_te)
        r2 = tool_fit_random_forest(self.X_tr, self.X_te, self.y_tr, self.y_te)
        r3 = tool_tabfm_zero_shot_predict(self.train_df, self.test_df, 'High_AI_Trust')
        return f"FINAL COMPARISON:\nLogistic Reg: {r1['roc_auc']} AUC\nRandom Forest: {r2['roc_auc']} AUC\nGoogle TabFM: {r3['roc_auc']} AUC"

agent = SurveyAnalysisReActAgent(df.iloc[:800], df.iloc[800:], X_train, X_test, y_train, y_test)
print(agent.run("Compare Logistic Regression, Random Forest, and Google TabFM Zero-Shot on AI Trust survey data."))

